In [ ]:
import logging

import torch

from omados.agents.rl import OBS_SIZE, RLAgent
from omados.env.game import SchafkopfEnv
from omados.models.policy_net import PolicyNet

# logging.basicConfig(level=logging.WARNING)  # suppress debug noise


# --- Load trained model ---
model = PolicyNet(obs_size=OBS_SIZE, hidden_size=256, depth=2)
model.load_state_dict(torch.load("../policy_net.pt", weights_only=True))
model.eval()

rl_agent = RLAgent(player_id=0, model=model)
agents = [
    rl_agent,
    RLAgent(player_id=1, model=model),
    RLAgent(player_id=2, model=model),
    RLAgent(player_id=3, model=model),
    # RandomAgent(player_id=1),
    # RandomAgent(player_id=2),
    # RandomAgent(player_id=3),
]
env = SchafkopfEnv(agents)

In [ ]:
# --- Run N games and collect stats ---
N = 1000
rl_wins = 0
rl_rewards = []

for _ in range(N):
    for a in agents:
        a.memory = []
    rl_agent.trajectory = []

    result = env.play_game()
    if result is None:
        continue  # Ramsch

    outcome, player_reward, opp_reward, contract, player_team, opp_team = result
    reward = player_reward if rl_agent.player_id in player_team else opp_reward
    rl_rewards.append(reward)
    if reward > 0:
        rl_wins += 1

played = len(rl_rewards)
print(f"Games played (non-Ramsch): {played}")
print(f"RL agent win rate:         {rl_wins / played:.1%}")
print(f"Mean reward:               {sum(rl_rewards) / played:+.4f}")
print("Random agent baseline:     ~50% win rate, ~0.0 mean reward")

In [ ]:
logging.basicConfig(level=logging.DEBUG)


for a in agents:
    a.memory = []


env.play_game()